# prenatalppkt walkthrough

Walks through prenatalppkt's pipeline on one real exam file, from the most basic checks (is this a real, readable file?) up to a complete patient record. Each section runs real code against real data and states plainly what the output means, including where something is missing or wrong.

Test case: `tests/data/Apple_Sally_pretty.json`, a real Observer JSON export.

## 1. The most basic checks

Before anything clever happens, the program needs to answer a few very simple questions: is this actually a file? Can it be opened? Is the text inside valid JSON? Does it describe at least one fetus with at least one measurement? If any of these fail, nothing downstream can work, so this is where any real exam file has to start.

In [10]:
import json
from pathlib import Path

DATA_PATH = Path("tests/data/Apple_Sally_pretty.json")

print("Does the file exist on disk?", DATA_PATH.exists())
print("Is it a real file (not a folder)?", DATA_PATH.is_file())

with open(DATA_PATH) as f:
    raw_text = f.read()
print(f"Read {len(raw_text)} characters of text from disk")

exam_data = json.loads(raw_text)
print("Parsed successfully as JSON. Top-level sections found:", list(exam_data.keys()))

fetuses = exam_data.get("fetuses", [])
print(f"\nNumber of fetuses described in this file: {len(fetuses)}")
assert len(fetuses) > 0, "nothing to check further if there are no fetuses"

measurements = fetuses[0].get("measurements", [])
print(f"Number of raw measurements recorded for fetus 1: {len(measurements)}")
for m in measurements:
    print(f"  {m.get('label')}: {m.get('value')} {m.get('unit_of_measure')}, at the {m.get('calculated_percentile')}th percentile")

Does the file exist on disk? True
Is it a real file (not a folder)? True
Read 50797 characters of text from disk
Parsed successfully as JSON. Top-level sections found: ['exam', 'adnexa', 'cervix', 'endomyocds', 'finalize', 'gyn_procedure', 'hist_phys_vitals', 'uterine_artery', 'uterus', 'fetuses']

Number of fetuses described in this file: 1
Number of raw measurements recorded for fetus 1: 6
  AC: 22.62 cm, at the 55.6th percentile
  BPD: 6.68 cm, at the 51.2th percentile
  HC: 25 cm, at the 42.5th percentile
  Femur: 5.01 cm, at the 46.8th percentile
  Nuchal Fold: 1 cm, at the 0th percentile
  Cerebellum: 3 cm, at the 0th percentile


## 2. Turning one raw measurement into a plain-English finding

A raw measurement like "AC: 22.62 cm, 55.6th percentile" doesn't mean anything to a computer by itself. The program compares it against a growth-reference table (`data/mappings/biometry_hpo_mappings.yaml`) that says, for each of 7 measurement types, which percentile ranges count as normal and which count as abnormal, and what the standard medical name (an HPO term) and standard lab code (a LOINC code) for that finding are.

There are 8 percentile ranges in the table (≤3rd, 3rd-5th, 5th-10th, 10th-50th, 50th-90th, 90th-95th, 95th-97th, and above 97th), and a measurement's raw percentile decides which one it falls into.

In [11]:
from prenatalppkt.etl.term_bin_factory import TermBinFactory

factory = TermBinFactory()
ac_measurement = next(m for m in measurements if m["label"] == "AC")

finding = factory.create_term_bin(
    name="AC",
    value_mm=ac_measurement["value"] * 10,
    percentile=ac_measurement["calculated_percentile"],
    method=None,
)
print(f"Raw value: {ac_measurement['value']} cm, at the {ac_measurement['calculated_percentile']}th percentile")
print(f"-> Plain-English name: {finding.hpo_label} ({finding.hpo_id})")
print(f"-> Is this reading normal? {finding.normal}")
print(f"-> Standard lab code (LOINC): {finding.loinc_code} - {finding.loinc_label}")

print("\nNow the same thing for every measurement on this exam, using the real extractor:")
from prenatalppkt.etl.extractors import observer as observer_extractor

all_findings = observer_extractor.extract_all_fetuses(exam_data)[1]
for f in all_findings:
    loinc_note = f.loinc_code or "(no lab code available for this measurement type)"
    print(f"  {f.description}  ->  {f.hpo_label} ({f.hpo_id})  |  LOINC: {loinc_note}")

DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for head_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for biparietal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for femur_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for abdominal_circumference
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for occipitofrontal_diameter
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for crown_rump_length
DEBUG:prenatalppkt.mapping_loader:Loaded 8 bins for nuchal_translucency
DEBUG:prenatalppkt.etl.term_bin_factory:Loaded mappings for: ['head_circumference', 'biparietal_diameter', 'femur_length', 'abdominal_circumference', 'occipitofrontal_diameter', 'crown_rump_length', 'nuchal_translucency']
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=AC, value=226.20000000000002mm, percentile=55.6%, ga=None, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0034207 - Abnormal fetal gastrointestinal system morphology
DEBUG:prenatalppkt.etl.t

Raw value: 22.62 cm, at the 55.6th percentile
-> Plain-English name: Abnormal fetal gastrointestinal system morphology (HP:0034207)
-> Is this reading normal? True
-> Standard lab code (LOINC): LOINC:11979-2 - Fetal Abdomen Circumference US

Now the same thing for every measurement on this exam, using the real extractor:
  AC: 226.2 mm (55.6%) at 26w6d [Fetus 1]  ->  Abnormal fetal gastrointestinal system morphology (HP:0034207)  |  LOINC: LOINC:11979-2
  BPD: 66.8 mm (51.2%) at 26w6d [Fetus 1]  ->  Abnormality of skull size (HP:0000240)  |  LOINC: LOINC:11820-8
  HC: 250.0 mm (42.5%) at 26w6d [Fetus 1]  ->  Abnormality of skull size (HP:0000240)  |  LOINC: LOINC:11984-2
  Femur: 50.1 mm (46.8%) at 27w0d [Fetus 1]  ->  Abnormal femur morphology (HP:0002823)  |  LOINC: LOINC:11963-6


**Known gap, not new today:** 3 of the 7 measurement types (crown-rump length, nuchal translucency, occipitofrontal diameter) never get a lab code - their entries in the growth-reference table are missing the LOINC block the other 4 have. A first-trimester-only exam (crown-rump length only) always shows 0 lab-coded measurements as a result, even though it still gets a correct plain-English finding. Already documented in `test_loinc_workflow_corpus.py`.

**Two real bugs found and fixed while checking this mechanism:**
1. `PercentileRange.evaluate()` was supposed to reject a percentile outside 0-100, but only checked the upper end - a negative number was silently treated as "below the 3rd percentile."
2. **More significant:** the function converting "26.9 weeks" into "26 weeks, 6 days" could silently lose a day from a floating-point rounding quirk (a value that should mean exactly "1 day" computed as 0.9999999999999964, truncated down to 0).

## 3. Reading the doctor's written notes for medical terms

Biometry numbers are structured data, so matching them to a growth-reference table is straightforward. A doctor's free-text note is not structured - it's just sentences. Finding medical terms inside plain English sentences is a different, much harder problem, handled by a separate tool called **fenominal**. It reads a block of text and tries to match phrases in it to the official Human Phenotype Ontology (HPO) vocabulary, including recognizing when a sentence says a finding is **absent** ("no evidence of X") rather than present.

Apple Sally's note is a good test case: it names one real finding by its exact official medical name, and separately rules out three other findings by name in the same sentence.

In [12]:
from prenatalppkt.hpo import HpoParser
from prenatalppkt.etl.sections import parse_clinical_impression
import gzip

HP_JSON_GZ = Path("tests/data/hp.json.gz")
TMP_HP_JSON = Path("/tmp/hp_walkthrough_v3.json")
with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
    with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
        f_out.write(f_in.read())
hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
hpo_cr = hpo_parser.get_hpo_concept_recognizer()

impression = parse_clinical_impression(exam_data, "observer_json", hpo_cr=hpo_cr)
print("The doctor's written note:")
print(impression["impression_text"])
print("\nMedical terms this tool found in that text:")
for t in impression["hpo_terms"]:
    status = "RULED OUT (the note says this is absent)" if t.excluded else "PRESENT"
    print(f"  {t.hpo_label} ({t.hpo_id}) - {status}")

DEBUG:hpotk.util:Using default encoding 'utf-8'
DEBUG:hpotk.util:Opening /tmp/hp_walkthrough_v3.json
DEBUG:hpotk.util:Looks like a local file: /tmp/hp_walkthrough_v3.json
DEBUG:hpotk.util:Looks like decompressed data
DEBUG:hpotk.ontology.load.obographs._load:Extracting ontology terms
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_requirement
DEBUG:hpotk.ontology.load.obographs._factory:Unknown synonym type http://purl.obolibrary.org/obo/hp#allelic_require

The doctor's written note:
Thank you for involving us in the care of the patient.nseled about the limitations of the exam. Although the absence of any sonographic markers reduces the likelihood of fetal aneuploidy, a normal ultrasound exam cannot exclude abnormal fetal genetics; definitive determination requires diagnostic genetic testing. 

Medical terms this tool found in that text:
  Macrocephaly (HP:0000256) - RULED OUT (the note says this is absent)
  Ventriculomegaly (HP:0002119) - RULED OUT (the note says this is absent)
  Agenesis of corpus callosum (HP:0001274) - PRESENT


**A real, honest gap.** The note also says, word for word, "This is consistent with a Dandy-Walker malformation" - the *exact* official medical name for HP:0001305. It never shows up above. Checked directly against the loaded medical dictionary: that HPO term exists, isn't outdated, and its official name matches the sentence character for character - fenominal simply doesn't recognize it, in this sentence or on its own, in several different phrasings tried.

This isn't a one-off. Checking every real exam file the same way (not just Apple Sally's) found the same pattern twice more: a real, present, named finding gets dropped from the output while other findings in the same note are recognized correctly. Testing the missed phrases *alone*, outside the full sentence, often works fine - so it looks like fenominal has more trouble with full natural sentences than with an isolated exact term. This is a limitation of the third-party tool, not a bug in this codebase, but it's a real one worth knowing about before trusting the output as complete.

Each of these confirmed gaps now has its own test in the test suite, written to fail on purpose (an ["xfail"](https://blog.ganssle.io/articles/2021/11/pytest-xfail.html) test) so the problem stays visible instead of being silently masked.

## 4. The anatomy scan section

Separate from the written note, the anatomy section records a structure-by-structure checklist (is the head normal, abnormal, or not seen?) plus short specific-finding notes when something is flagged abnormal. Those short notes go through the same fenominal tool as the written note above.

In [13]:
from prenatalppkt.etl.sections import parse_fetal_anatomy

anatomy = parse_fetal_anatomy(exam_data, "observer_json", hpo_cr=hpo_cr)
print("Body parts checked and marked normal:", anatomy["normal_structures"])
print("\nBody parts marked abnormal:", anatomy["abnormal_structures"])
print("Specific problems noted:", anatomy["anomalies"])
print("\nMedical terms found from this section:")
for t in anatomy["hpo_terms"]:
    status = "RULED OUT (absent)" if t.excluded else "PRESENT"
    print(f"  {t.hpo_label} ({t.hpo_id}) - {status}")

Body parts checked and marked normal: ['Face/Neck', 'Neck', 'Nuchal Fold', 'Profile', 'Orbits', 'Nose/Lips', 'Palate', 'Face', 'Th. Cav.', 'Diaphragm', 'Heart', 'Four Chamber View', 'Proximal Left Outflow', 'Proximal Right Outflow', 'Short Axis of Greater Vessels', 'Cardiac Axis', 'Interventricular Septum', 'Interatrial Septum', 'Cardiac Position', 'Abd. Cav.', 'Stomach', 'Right Kidney', 'Left Kidney', 'Bladder', 'Abd. Wall', 'Spine', 'Cervical Spine', 'Thoracic Spine', 'Lumbar Spine', 'Sacrum', 'Extrems', 'Lt Humerus', 'Rt Humerus', 'Lt Forearm', 'Rt Forearm', 'Lt Hand', 'Rt Hand', 'Lt Femur', 'Rt Femur', 'Lt Low Leg', 'Rt Low Leg', 'Lt Foot', 'Rt Foot', 'Genitalia', 'Placenta', 'Umbl. Cord', 'PCI']

Body parts marked abnormal: ['Head', 'Cerebellum']
Specific problems noted: [{'structure': 'Head', 'description': 'Dandy Walker', 'variant_type': 'Abnormal'}]

Medical terms found from this section:
  Neural tube defect (HP:0045005) - RULED OUT (absent)


Notice the specific problem noted is literally "Dandy Walker" (two words, no "malformation") - the terse, structured version of the same finding that was missed above. It's missed here too. Interestingly, this terse-phrase path works correctly for every *other* real exam file checked (a kidney problem, an abdominal wall problem, a missing-skull problem, and a heart problem all come through correctly this way) - so this looks specific to this exact term, not a general problem with short structured notes.

## 5. What we actually confirmed by hand, across every real exam

Checking one file only proves one file works. Before trusting this pipeline, every one of the 5 real Observer exam files (and every real ViewPoint HL7 message, and the one real gynecology exam file) was checked by hand: what does the raw text say, and does the program produce exactly the medical terms that text describes - no more, no fewer?

Here's the complete result table.

In [14]:
ground_truth = [
    ("Apple", "Macrocephaly - ruled out", "correct"),
    ("Apple", "Ventriculomegaly - ruled out", "correct"),
    ("Apple", "Agenesis of corpus callosum - present", "correct"),
    ("Apple", "Neural tube defect - ruled out", "correct"),
    ("Apple", "Dandy-Walker malformation - present", "MISSED by fenominal"),
    ("Blue", "Unilateral renal agenesis - present", "correct"),
    ("Blue", "Bicornuate uterus - present", "correct"),
    ("Blue", "Renal agenesis - present (anatomy section)", "correct"),
    ("Blue", "Unicornuate uterus - present", "MISSED by fenominal"),
    ("Charm", "Abdominal wall defect - present", "correct"),
    ("Charm", "Omphalocele - present", "correct"),
    ("Diva", "Acrania - present", "correct"),
    ("Eclair", "Abnormal heart morphology (generic) - present", "correct"),
    ("Eclair", "Hypoplastic left ventricle - present (anatomy section)", "correct"),
    ("Eclair", "Enlarged right heart chambers - present (specific)", "MISSED by fenominal"),
]

print(f"{'Patient':<8} {'Finding':<52} {'Result'}")
print("-" * 80)
for patient, finding, result in ground_truth:
    print(f"{patient:<8} {finding:<52} {result}")

correct = sum(1 for _, _, r in ground_truth if r == "correct")
print(f"\n{correct}/{len(ground_truth)} confirmed findings come through correctly.")
print(f"{len(ground_truth) - correct} confirmed real findings are silently missed, all by the same tool (fenominal), all in full-sentence context.")

Patient  Finding                                              Result
--------------------------------------------------------------------------------
Apple    Macrocephaly - ruled out                             correct
Apple    Ventriculomegaly - ruled out                         correct
Apple    Agenesis of corpus callosum - present                correct
Apple    Neural tube defect - ruled out                       correct
Apple    Dandy-Walker malformation - present                  MISSED by fenominal
Blue     Unilateral renal agenesis - present                  correct
Blue     Bicornuate uterus - present                          correct
Blue     Renal agenesis - present (anatomy section)           correct
Blue     Unicornuate uterus - present                         MISSED by fenominal
Charm    Abdominal wall defect - present                      correct
Charm    Omphalocele - present                                correct
Diva     Acrania - present                              

## 6. The other exam-level sections

A few more sections round out the exam: why it was ordered, when the pregnancy is dated to, the estimated fetal weight, and how the measurements compare to each other proportionally. These are simpler, mostly-numeric sections with less room for the kind of gap seen above.

In [15]:
from prenatalppkt.etl.sections import (
    parse_clinical_indication,
    parse_pregnancy_dating,
    parse_estimated_fetal_weight,
    parse_fetal_ratios,
)

indication = parse_clinical_indication(exam_data, "observer_json")
print("Why the exam was ordered:", indication.get("indication_text") or "(not recorded in this file)")

dating = parse_pregnancy_dating(exam_data, "observer_json")
print(f"Pregnancy dating: last period={dating.get('lmp')} due date={dating.get('edd')}")

efw = parse_estimated_fetal_weight(exam_data, "observer_json")
print(f"Estimated fetal weight: {efw.get('efw_grams')}g, {efw.get('percentile')}th percentile, category={efw.get('growth_category')}")

ratios = parse_fetal_ratios(exam_data, "observer_json")
print("How the measurements compare to each other:")
for r in ratios["ratios"]:
    print(f"  {r['name']}: {r['value']} (expected {r['expected_range']}) - within range: {r['within_range']}")

Why the exam was ordered: (not recorded in this file)
Pregnancy dating: last period=0001-01-01 due date=None
Estimated fetal weight: 1014.8g, 55.6th percentile, category=AGA
How the measurements compare to each other:
  HC/AC: 1.105 (expected (1.04, 1.22)) - within range: True
  FL/AC: 22.149 (expected (20.0, 24.0)) - within range: True
  FL/BPD: 75 (expected (71.0, 87.0)) - within range: True


## 7. Building the complete official record

Everything above - biometry, the written note, the anatomy checklist - gets stitched together into one official, standardized patient record (called a "Phenopacket," a shared format used across genomics and rare-disease research so records from different hospitals can be compared). This is the single function call a real user of this program would actually make.

In [16]:
from prenatalppkt.builders import build_observer_phenopacket
from datetime import datetime, timezone
from google.protobuf.timestamp_pb2 import Timestamp
from google.protobuf.json_format import MessageToJson, Parse
import phenopackets.schema.v2 as pps2

now_ts = Timestamp()
now_ts.FromDatetime(datetime.now(tz=timezone.utc))

phenopackets = build_observer_phenopacket(exam_data, hpo_parser, now_ts, accession_id="apple-sally")
record = phenopackets[0]
print(f"Built {len(phenopackets)} record(s). This one has {len(record.phenotypic_features)} findings:")
for finding in record.phenotypic_features:
    status = "ruled out" if finding.excluded else "present"
    print(f"  {finding.type.label} ({finding.type.id}) - {status}")

# Confirm the record is valid and nothing was lost saving/reloading it
json_str = MessageToJson(record)
reloaded = Parse(json_str, pps2.Phenopacket())
assert reloaded == record
print("\nThe record is valid and nothing was lost saving it to a file and reading it back.")

DEBUG:prenatalppkt.etl.extractors.observer:Starting Observer JSON extraction (multi-fetus)
DEBUG:prenatalppkt.etl.extractors.observer:Processing fetus 1, scan_type=t2_t3_biometry
DEBUG:prenatalppkt.etl.extractors.observer:Found 6 measurements
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: AC
DEBUG:prenatalppkt.etl.extractors.observer:AC has percentile=55.6% (valid)
DEBUG:prenatalppkt.etl.extractors.observer:Creating TermBin for AC: value=226.20000000000002mm, percentile=55.6%, ga=<GestationalAge: 26 weeks, 6 days>
DEBUG:prenatalppkt.etl.term_bin_factory:Creating TermBin: name=AC, value=226.20000000000002mm, percentile=55.6%, ga=<GestationalAge: 26 weeks, 6 days>, method=None
DEBUG:prenatalppkt.etl.term_bin_factory:Selected HPO: HP:0034207 - Abnormal fetal gastrointestinal system morphology
DEBUG:prenatalppkt.etl.term_bin_factory:Created TermBin: HP:0034207 - normal=True
DEBUG:prenatalppkt.etl.extractors.observer:Processing measurement: BPD
DEBUG:prenatalppkt.etl.ext

Built 1 record(s). This one has 7 findings:
  Abnormal fetal gastrointestinal system morphology (HP:0034207) - ruled out
  Abnormality of skull size (HP:0000240) - ruled out
  Abnormal femur morphology (HP:0002823) - ruled out
  Macrocephaly (HP:0000256) - ruled out
  Ventriculomegaly (HP:0002119) - ruled out
  Agenesis of corpus callosum (HP:0001274) - present
  Neural tube defect (HP:0045005) - ruled out

The record is valid and nothing was lost saving it to a file and reading it back.


## 8. Attaching a genetics file

A Phenopacket can carry more than the ultrasound findings above - it can also carry genetic sequencing results, so the clinical picture and the genetics travel together. This step attaches a genetics file (VCF format) and lists what variants are in it.

**Important limit to be honest about:** this step only attaches the file and lists its contents. It does **not** check whether any of the listed variants actually explain the findings above, and does **not** do any real genetic interpretation. It proves the record *can* carry genetics data - it doesn't yet do anything clinically meaningful with it.

In [17]:
from prenatalppkt.genomics import scan_vcf_file, build_vcf_file_entry, build_genomic_interpretation

vcf_path = Path("tests/data/Apple_Sally.vcf")
variants = scan_vcf_file(vcf_path)
print(f"Found {len(variants)} variant location(s) in the genetics file:")
for v in variants:
    print(f"  {v.chrom}:{v.pos} {v.ref}->{v.alt} (genome build {v.genome_assembly})")

record.files.append(
    build_vcf_file_entry(
        vcf_path.resolve().as_uri(),
        attributes={"genomeAssembly": variants[0].genome_assembly},
    )
)
record.interpretations.append(
    build_genomic_interpretation(
        variants, subject_id=record.subject.id, interpretation_id=f"{record.id}-genomic-interp-1"
    )
)
print(f"\nThe record now carries {len(record.files)} genetics file(s) and {len(record.interpretations)} interpretation entry.")

TMP_HP_JSON.unlink(missing_ok=True)

Found 3 variant location(s) in the genetics file:
  chr1:1000 A->G (genome build GRCh38)
  chr7:200000 C->T (genome build GRCh38)
  chrX:500000 G->A (genome build GRCh38)

The record now carries 1 genetics file(s) and 1 interpretation entry.


## 9. A second, older way of doing this that isn't used anymore

There's a second, separate, older set of code that also claims to build a patient record from measurements - `PhenotypicExporter`, `TermObservation`, `SonographicMeasurement`, from around November-December 2025.

It doesn't read real exam files, doesn't use fenominal, and builds its output by hand as a plain data structure rather than a real Phenopacket. Nothing in the pipeline above depends on it - a leftover from before a rewrite replaced it with everything shown here. Its main working path is also currently broken (a function it depends on was renamed and never updated) - unnoticed because nothing calls that path. Retiring it is tracked separately, not decided here.

## 10. The test suite

**698 passing tests**, 21 skipped (real gaps needing a clinical decision, not oversights), **7 expected to fail on purpose**.

That last category: when this audit found fenominal misses a real finding, the response was a test that names exactly which finding is missing and expects it to fail (`xfail`) - a clearly-labeled expected failure, not a silent gap. If fenominal ever catches it, the test unexpectedly passes, which itself gets flagged.

| Area | Where the tests are |
|---|---|
| Growth-reference lookup (percentile -> HPO + LOINC) | `test_mapping_loader.py`, `test_percentile_range.py`, `test_reference_range.py`, `test_term_bin.py` |
| Gestational age math | `test_gestational_age.py` |
| Reading exam files (Observer + ViewPoint) | `etl/extractors/` |
| Doctor's notes + anatomy text -> HPO (fenominal) | `hpo/`, `etl/sections/test_clinical_impression.py`, `etl/sections/test_fetal_anatomy.py` |
| Building the complete record | `builders/` |
| Genetics file attachment | `genomics/` |

**Not built yet** (each returns empty data, not wrong data): maternal history, placenta, amniotic fluid, umbilical cord, and cardiac echo detail (general heart findings work via the anatomy path in Section 4/5 - it's the *detailed* echo exam that isn't wired in).

## 11. Summary

**Solid, directly verified:**
- Reading a real exam file, validating it, pulling out measurements
- Measurement -> medical finding + lab code, for the 4 core measurement types, bin-boundary math directly tested
- Building the complete patient record, round-tripping through JSON without loss
- Attaching a genetics file (structurally - not interpreting it)

**Known, real gaps:**
- 3 of 7 measurement types never get a lab code (data gap, not a design problem)
- fenominal misses some real, clearly-named findings in full sentences - 4 confirmed examples, each with an `xfail` test
- Cardiac echo detail, maternal history, placenta, amniotic fluid, umbilical cord not built yet
- Only one first-trimester case (Diva) checked so far - deserves its own closer look

**Fixed today:**
- Gestational-age conversion bug that could silently lose a day, up to 43% of cases
- Percentile-boundary function not rejecting invalid negative input
- A notebook cell crashing on a file it wasn't supposed to touch